# Etapa 2 — Modelagem e Avaliação da MLP

Notebook experimental autocontido para a Etapa 2 do Tech Challenge: construção da MLP em PyTorch, treino com batching e early stopping, comparação com baselines, análise de custo e registro final no MLflow.

Este notebook evita imports de `src/` de propósito: a etapa é exploratória e deve permitir experimentar arquitetura, parâmetros e thresholds antes de refatorar o código de produção.

## 1. Imports, paths, seeds e MLflow

Configura ambiente, fixa seeds e define os diretórios usados para dados, artefatos e tracking local do MLflow.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from mlflow.models import infer_signature
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "etapa2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
EXPERIMENT_NAME = "telco-churn-mlp-notebooks"
MLFLOW_TRACKING_DIR = NOTEBOOKS_DIR / "mlruns"
MLFLOW_TRACKING_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seeds(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


set_seeds(RANDOM_SEED)
print(f"project_root={PROJECT_ROOT}")
print(f"device={DEVICE}")
print(f"mlflow_tracking_uri={mlflow.get_tracking_uri()}")

## 2–3. Carga dos XLSX, target, leakage e split

Replica inline a união das tabelas brutas. Depois cria `target`, remove IDs/leakage e faz split estratificado 64/16/20.

In [ ]:
RAW_DATA_FILES = {
    "demographics": "Telco_customer_churn_demographics.xlsx",
    "location": "Telco_customer_churn_location.xlsx",
    "services": "Telco_customer_churn_services.xlsx",
    "population": "Telco_customer_churn_population.xlsx",
    "status": "Telco_customer_churn_status.xlsx",
}
ID_COLUMNS = ("CustomerID", "ID")
LEAKAGE_COLUMNS = (
    "ChurnLabel",
    "ChurnValue",
    "CustomerStatus",
    "ChurnScore",
    "ChurnScoreCategory",
    "ChurnCategory",
    "ChurnReason",
)
TARGET_COLUMN = "target"


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    renamed = {
        column: str(column).strip().replace(" ", "").replace("_", "")
        for column in df.columns
    }
    return df.rename(columns=renamed)


def load_raw_tables(data_dir: Path = DATA_DIR) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    for table_name, file_name in RAW_DATA_FILES.items():
        path = data_dir / file_name
        table = pd.read_excel(path, sheet_name=0)
        tables[table_name] = clean_column_names(table).drop(columns=["Count"], errors="ignore")
    return tables


def load_telco_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    tables = load_raw_tables(data_dir)
    dataset = (
        tables["demographics"]
        .merge(tables["location"], on="CustomerID")
        .merge(tables["services"], on="CustomerID")
        .merge(tables["population"], on="ZipCode")
        .merge(tables["status"], on="CustomerID")
        .drop(columns=["ID"], errors="ignore")
    )
    if "ChurnValue" not in dataset.columns:
        raise ValueError("Coluna ChurnValue não encontrada para construir o target.")
    dataset[TARGET_COLUMN] = (dataset["ChurnValue"] > 0).astype(int)
    return dataset


df = load_telco_dataset(DATA_DIR)
columns_to_drop = list(ID_COLUMNS) + list(LEAKAGE_COLUMNS) + [TARGET_COLUMN]
X = df.drop(columns=columns_to_drop, errors="ignore")
y = df[TARGET_COLUMN]

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=VAL_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_SEED,
)
x_test_raw = x_test.copy()
y_test_array = y_test.to_numpy()

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(x_train), "churn_rate": y_train.mean()},
        {"split": "validation", "rows": len(x_val), "churn_rate": y_val.mean()},
        {"split": "test", "rows": len(x_test), "churn_rate": y_test.mean()},
    ]
)
print(f"dataset_shape={df.shape}")
print(f"feature_shape={X.shape}")
split_summary.style.format({"churn_rate": "{:.2%}"})

## 4. Pré-processamento e DataLoaders

Imputa e escala variáveis numéricas, imputa e aplica one-hot nas categóricas. O `fit` do pré-processador acontece apenas no treino.

In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(x: pd.DataFrame) -> ColumnTransformer:
    numeric_features = x.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = x.select_dtypes(exclude=["number", "bool"]).columns.tolist()
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )


def as_float_tensor(values: Any) -> torch.Tensor:
    if hasattr(values, "toarray"):
        values = values.toarray()
    return torch.tensor(values, dtype=torch.float32)


preprocessor = build_preprocessor(x_train)
x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed = preprocessor.transform(x_val)
x_test_processed = preprocessor.transform(x_test)

x_train_tensor = as_float_tensor(x_train_processed)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
x_val_tensor = as_float_tensor(x_val_processed)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1, 1)
x_test_tensor = as_float_tensor(x_test_processed)
INPUT_DIM = x_train_tensor.shape[1]


def make_loaders(batch_size: int) -> tuple[DataLoader, DataLoader]:
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
    )


numeric_features = x_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = x_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()
try:
    feature_names = preprocessor.get_feature_names_out().tolist()
except Exception:
    feature_names = [f"feature_{i}" for i in range(INPUT_DIM)]

feature_summary = pd.DataFrame(
    [
        {"feature_group": "numeric_original", "count": len(numeric_features)},
        {"feature_group": "categorical_original", "count": len(categorical_features)},
        {"feature_group": "processed_input_dim", "count": INPUT_DIM},
    ]
)
feature_summary